# Exploratory Data Analysis

In [28]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
COHORTS = ["LUSC"]

In [31]:
lusc_dir = "../data/raw/LUSC"
first_dir = os.listdir(lusc_dir)[0]
files = [f for f in os.listdir(f"{lusc_dir}/{first_dir}") if f.endswith(".tsv")]
sample_tsv = pd.read_csv(f"{lusc_dir}/{first_dir}/{files[0]}", sep="\t", comment="#")


# build ENSG -> gene_name mapping from this one file
gene_map = sample_tsv[["gene_id", "gene_name"]].dropna()
gene_map["gene_id_clean"] = gene_map["gene_id"].str.split(".").str[0]  # remove version suffix
gene_map = gene_map.set_index("gene_id_clean")["gene_name"].to_dict()

# your gene columns have version suffixes too — strip them
def to_name(ensg):
    clean = ensg.split(".")[0]
    return gene_map.get(clean, ensg)  # fallback to ENSG if not found

In [10]:
import plotly.io as pio
pio.templates.default = "catppuccin"

In [11]:
cohort = COHORTS[0]
df = pd.read_parquet(f"{PROCESSED_DIR}/{cohort}.parquet")
tss_summary = df.groupby("tss").agg(
    patients=("patient_barcode", "count"),
    os_events=("OS", "sum"),
    pfi_events=("PFI", "sum")
).reset_index()
tss_summary["os_rate"] = (tss_summary["os_events"] / tss_summary["patients"]).round(2)
print(tss_summary.sort_values("patients", ascending=False).to_string())

   tss  patients  os_events  pfi_events  os_rate
19  85        50       17.0        12.0     0.34
18  77        44       27.0        15.0     0.61
11  56        40       12.0        10.0     0.30
15  66        39        9.0         3.0     0.23
2   22        37       35.0        16.0     0.95
13  60        25        9.0         8.0     0.36
7   43        22        7.0         5.0     0.32
14  63        21        2.0         5.0     0.10
3   33        20       17.0         9.0     0.85
1   21        18        9.0         8.0     0.50
0   18        18       12.0         5.0     0.67
4   34        18        9.0         4.0     0.50
6   39        17        9.0         8.0     0.53
29  NC        16        6.0         5.0     0.38
24  98        14        5.0         5.0     0.36
5   37        13        1.0         1.0     0.08
12  58        12        5.0         4.0     0.42
22  94         9        1.0         4.0     0.11
20  90         8        1.0         2.0     0.12
16  68         7    

In [13]:
print(tss_plot["tss"].tolist())

['22', '33', '18', '77', '39', '21', '34', '58', 'NC', '60', '98', '85', '43', '56', '66', '63', '37']


In [15]:
import plotly.express as px
tss_plot = tss_summary[tss_summary["patients"] >= 10].sort_values("os_rate", ascending=False)
tss_plot["tss"] = tss_plot["tss"].astype(str)

fig = px.bar(
    tss_plot,
    x="tss", y="os_rate",
    title="Death rate by tissue source site — LUSC",
    labels={"tss": "TSS (site code)", "os_rate": "OS death rate"},
)
fig.update_layout(xaxis=dict(type="category", categoryorder="array", categoryarray=tss_plot["tss"].tolist()))
fig.show()

The survival rates varies drastically across sites. TSS 22 has 93% death rate and TSS 37 has 8% -> That's a 85 percentage point spread across the hospitals. These differences could either be because of:
- Real biological difference - Certain sites get more later stage cancers than others
- Follow up artefact
- Both

A large amount of variation in the death rate could be explained purely by which site the patient went through. Our model should be able to absorb these differences so that only the biologically relevant covariates remain

In [16]:
from sklearn.decomposition import PCA

gene_cols = [c for c in df.columns if c.startswith("ENSG")]

pca = PCA(n_components=2)
coords = pca.fit_transform(df[gene_cols])

pca_df = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "tss": df["tss"].astype(str),
    "patient": df["patient_barcode"]
})

fig = px.scatter(
    pca_df, x="PC1", y="PC2", color="tss",
    hover_data=["patient"],
    title=f"PCA of gene expression by TSS — LUSC | PC1: {pca.explained_variance_ratio_[0]:.1%}, PC2: {pca.explained_variance_ratio_[1]:.1%}",
)
fig.show()

There's a good mix of gene expression by TSS. So the sites don't drive the first and second most important principle components

In [17]:
pca3 = PCA(n_components=3)
coords3 = pca3.fit_transform(df[gene_cols])

pca_df3 = pd.DataFrame({
    "PC1": coords3[:, 0],
    "PC2": coords3[:, 1],
    "PC3": coords3[:, 2],
    "tss": df["tss"].astype(str),
    "patient": df["patient_barcode"]
})

fig1 = px.scatter(
    pca_df3, x="PC1", y="PC3", color="tss",
    hover_data=["patient"],
    title=f"PC1 vs PC3 | PC1: {pca3.explained_variance_ratio_[0]:.1%}, PC3: {pca3.explained_variance_ratio_[2]:.1%}",
)
fig1.show()

fig2 = px.scatter(
    pca_df3, x="PC2", y="PC3", color="tss",
    hover_data=["patient"],
    title=f"PC2 vs PC3 | PC2: {pca3.explained_variance_ratio_[1]:.1%}, PC3: {pca3.explained_variance_ratio_[2]:.1%}",
)
fig2.show()

In PC2 vs. PC3, you can see a few outliers are all orange (site 37)

The clear conclusion from these are: site effects in LUSC don't dominate gene expression variance. The variability is more subtle

In [18]:
from lifelines import KaplanMeierFitter
import plotly.graph_objects as go

fig = go.Figure()

for tss in df["tss"].unique():
    mask = df["tss"] == tss
    subset = df[mask].dropna(subset=["OS", "OS_time"])
    if len(subset) < 10:
        continue
    
    kmf = KaplanMeierFitter()
    kmf.fit(subset["OS_time"], event_observed=subset["OS"], label=str(tss))
    
    fig.add_trace(go.Scatter(
        x=kmf.timeline,
        y=kmf.survival_function_.values.flatten(),
        mode="lines",
        name=str(tss)
    ))

fig.update_layout(
    title="Kaplan-Meier survival curves by TSS — LUSC",
    xaxis_title="Days",
    yaxis_title="Survival probability",
    template="catppuccin"
)
fig.show()

By day 1000

- Some sites are still at ~0.8 survival probability (TSS 21, 63 — the high flat lines)
- Others are already at ~0.2 (TSS 77, the green line dropping steeply)

TSS 37 (orange) drops to zero around day 1000 — all patients dead. TSS 21 (blue) stays flat and high — very few deaths, long follow-up.

The flat tails on many curves are censored patients — the curve stops stepping down because no more deaths were observed, not because everyone survived.

In [32]:
gene_cols = [c for c in df.columns if c.startswith("ENSG")]

# 1. Distribution of a single gene
top_var_genes = df[gene_cols].var().sort_values(ascending=False)
top_gene = top_var_genes.index[0]

name = to_name(top_gene)

fig1 = px.histogram(
    df, x=top_gene, nbins=50,
    title=f"Z-score distribution — {name} (highest variance gene)",
    labels={top_gene: "Z-score"}
)
fig1.show()

Bulk of the patients lie between -2 and 2. There are a few outliers but this is not alarming - it is common for some tumors to have extreme expressions of certain genes

In [33]:

# 2. Mean expression per TSS for top 10 genes
top10_genes = top_var_genes.index[:10].tolist()
tss_expr = df.groupby("tss")[top10_genes].mean().reset_index()
tss_expr_long = tss_expr.melt(id_vars="tss", var_name="gene", value_name="mean_zscore")

tss_expr_long["gene_name"] = tss_expr_long["gene"].apply(to_name)

fig3 = px.bar(
    tss_expr_long, x="tss", y="mean_zscore", color="gene_name",
    barmode="group",
    title="Mean expression per TSS — top 10 variance genes",
    labels={"tss": "TSS", "mean_zscore": "Mean z-score"}
)
fig3.show()


This graph is interesting - you can see site 37 has an unusually large expression for gene VPS72. And you can see other cases where there are big spikes of particular genes concentrated around certain sites

HDGF (red/pink) spikes at TSS ~51. HDGF is actually a known growth factor involved in cell proliferation. it has legitimate biological reasons to be prognostic. Whether its signal survives site adjustment will be an interesting result.

In [34]:
from scipy.stats import spearmanr

# compute spearman correlation of each gene with OS_time
# only use uncensored patients (OS=1) for a cleaner signal
died = df[df["OS"] == 1]

correlations = {}
for gene in gene_cols:
    r, p = spearmanr(died[gene], died["OS_time"])
    correlations[gene] = {"r": r, "p": p}

corr_df = pd.DataFrame(correlations).T.reset_index()
corr_df.columns = ["gene", "r", "p"]
corr_df = corr_df.sort_values("r", key=abs, ascending=False).head(20)

corr_df["gene_name"] = corr_df["gene"].apply(to_name)

fig = px.bar(
    corr_df, x="gene_name", y="r",
    title="Top 20 genes by Spearman correlation with OS time (deaths only)",
    labels={"gene_name": "Gene", "r": "Spearman r"},
    color="r",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0
)
fig.update_xaxes(tickangle=45)
fig.show()

Negative correlations (red — higher expression = shorter survival):

ADAMTSL4 (r ≈ -0.28) — extracellular matrix gene, could relate to invasion/metastasis hallmark
MAFK — transcription factor involved in stress response
FBXO2, MAP3K6, FLNC, SPOCD1 — various signalling and structural genes

Positive correlations (blue — higher expression = longer survival):

MYLIP (r ≈ +0.26) — involved in cholesterol metabolism
MORC4, PPP4R3B, UGT8 — DNA repair and metabolism
LMNB1, TMEM260, USP39 — nuclear structure and RNA splicing

Correlations are weak -> between -0.3 and 0.2

Both directions exist. For some genes, more expression means more survival, and for others, more expression means less survival. This is biologically plausible



In [41]:
top50_genes = top_var_genes.index[:50].tolist()
top50_names = [to_name(g) for g in top50_genes]
corr = df[top50_genes].corr()

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
import numpy as np

# cluster genes by correlation
dist = 1 - corr.abs()
dist = (dist + dist.T) / 2  # ensure symmetry
dist_vals = dist.values.copy()
np.fill_diagonal(dist_vals, 0)
linkage_matrix = linkage(squareform(dist_vals), method="ward")
order = leaves_list(linkage_matrix)

# reorder
ordered_genes = [top50_genes[i] for i in order]
ordered_names = [to_name(g) for g in ordered_genes]
corr_ordered = corr.loc[ordered_genes, ordered_genes]

fig = go.Figure(go.Heatmap(
    z=corr_ordered.values,
    x=ordered_names,
    y=ordered_names,
    colorscale="RdBu",
    zmin=-1, zmax=1,
    colorbar=dict(title="r")
))
fig.update_layout(
    title="Correlation heatmap — top 50 genes (clustered)",
    template="catppuccin",
    width=900, height=900,
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9))
)
fig.show()

Bottom-left cluster (HDGF, SF3B4, PIP5K1A, VPS72, MRPL9, PIEZO1, UNC50, LUZP1, TAF7, HINT1, DCTN2, YEATS4, RUSC2, CYTH3) — strongly positively correlated with each other (blue block), and negatively correlated with the top-right cluster (red).

Top-right cluster (MKI67, DUSP23, ROM1, LRIG3, VCIP1, RNF187) — a smaller co-expression module negatively correlated with the bottom-left group.
Middle section — mostly independent genes with weak correlations.

This is actually a meaningful biological finding. MKI67 is a well-known proliferation marker — its negative correlation with the HDGF/SF3B4 cluster suggests these two groups represent different tumour phenotypes. Some tumours are highly proliferative (high MKI67), others have high HDGF/SF3B4 expression — and these phenotypes are mutually exclusive.